<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.1-poisson/Ex07.1_01_slot_soft_bc.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.1 · Notebook 01 — The Wall Condition as a **Penalty**

**Paired with L7.1 · Fundamentals of PINNs**

The slot from notebook 00, solved by a network. The walls are held at the iron
temperature, and in this notebook that condition enters the loss as a term to
be minimised alongside the PDE:

$$\mathcal{L} = \underbrace{\frac{1}{N_f}\sum \bigl(k_{\mathrm{eff}}\nabla^2\hat\theta + q\bigr)^2}_{\text{physics}}
\;+\; w\,\underbrace{\frac{1}{N_b}\sum \hat\theta^2}_{\text{walls}}$$

This is **soft** enforcement. The network is asked to satisfy the walls; it is
not required to. Notebook 02 requires it, and the pair is the whole point of
L7.1.

## What you will do

1. Write the PDE residual with automatic differentiation.
2. Assemble the two-term loss and train with Adam, then L-BFGS.
3. Measure the error against the exact solution — in kelvin, not just as a norm.
4. Sweep the penalty weight `w` and find out that it matters more than you
   would like.

## One thing to watch

The two terms have different units and wildly different magnitudes. The
physics residual is in W/m³ and squares to something near 10¹²; the wall
residual is in K² and is of order 1. A loss that adds them raw is dominated by
whichever happens to be larger, which is a property of your unit system rather
than of your engineering. Section 2 does something about that.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.1-poisson/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The residual

$\mathcal{F} = k_{\mathrm{eff}}\,(\hat\theta_{,xx} + \hat\theta_{,yy}) + q$.

`d2(u, xy, 0)` is $\hat\theta_{,xx}$ and `d2(u, xy, 1)` is $\hat\theta_{,yy}$.
`pb.source` takes the two coordinate columns and works on tensors unchanged —
the manufactured source is polynomial, so there is no NumPy in it to trip over.

### Your turn

In [ ]:
# TODO: write the PDE residual.
#
#   def pde_residual(model, xy):
#       theta = model(xy)
#       theta_xx = d2(theta, xy, 0)
#       theta_yy = d2(theta, xy, 1)
#       q = pb.source(xy[:, 0:1], xy[:, 1:2])
#       return pb.K_EFF * (theta_xx + theta_yy) + q
#
# Note the 0:1 and 1:2 slicing. xy[:, 0] would give a flat (N,) tensor and the
# arithmetic would broadcast into an (N, N) residual that trains to something
# meaningless without ever raising.

raise NotImplementedError("Write pde_residual(model, xy)")

## 2 · The loss, and the units problem

Scale the physics residual by the peak source before squaring. Then both terms
are dimensionless, both are order one, and `w` means what you think it means —
the relative importance of the walls, rather than a number that silently
encodes the conversion from watts to kelvin.

This is the same argument L6.1 makes about $\lambda$ in a multi-term loss, and
the same one that returns in every remaining exercise of Part 2:
**non-dimensionalise each residual before choosing its weight.**

### Your turn

In [ ]:
# TODO: build the two-term loss.
#
#   Q_SCALE = float(pb.source(np.array([0.0]), np.array([0.0])))   # W/m^3
#
#   def make_loss(model, xy_f, xy_b, w):
#       def loss():
#           f = pde_residual(model, xy_f) / Q_SCALE      # dimensionless
#           b = model(xy_b)                              # theta on the wall, K
#           return mse(f) + w * mse(b)
#       return loss
#
#   The prescribed wall value is zero, so the wall residual is just the network
#   output there. Keep the two terms separate in your head: you will want to
#   report them apart in notebook 03.

raise NotImplementedError("Write make_loss(model, xy_f, xy_b, w)")

## 3 · Train

Adam to get close, L-BFGS to finish. Ex_06 notebook 02 measured why; here you
use it.

The points are sampled **once, before training**. L-BFGS evaluates the loss
several times per step during its line search and assumes it is looking at the
same function each time — resampling inside the loop breaks that assumption
and the line search fails in ways that are hard to read.

In [ ]:
N_COLL, N_WALL, W_DBC = 2000, 80, 1.0

set_seed(88)
model_soft = MLP(n_in=2, n_hidden=32, n_layers=4)
describe(model_soft, N_COLL)

xy_f = to_tensor(interior_points(N_COLL, pb.DOMAIN, seed=1), requires_grad=True)
xy_b = to_tensor(boundary_points(N_WALL, pb.DOMAIN, seed=1))

history_soft = train_two_stage(model_soft, make_loss(model_soft, xy_f, xy_b, W_DBC),
                               adam_steps=3000, lbfgs_steps=150, lr=1e-3)
plot_curves(history_soft, title="soft enforcement, w = 1")
plt.show()

**What you should see.** A loss that falls several orders under Adam and then
drops sharply again when L-BFGS takes over. The `collocation` line from
`describe` should read comfortably above 1 — if you have more parameters than
sample points, the network can satisfy the equation at every point you tested
and do anything at all between them, and there is no held-out set to catch it.

---

## 4 · How wrong is it, in kelvin?

A relative $L^2$ norm is the standard PINN score and it is dimensionless, which
makes it comparable between problems. It is also the number most likely to
flatter you. Report the largest single error in kelvin beside it, because that
is what a machine designer is actually asking about — and it is usually at the
hot spot.

### Your turn

In [ ]:
# TODO: score the model on a grid, in both currencies.
#
#   X, Y, pts = grid_points(161, 161, pb.DOMAIN)
#   with torch.no_grad():
#       theta_hat = to_numpy(model_soft(to_tensor(pts))).ravel()
#   theta_ref = pb.theta_exact(pts[:, 0], pts[:, 1])
#
#   Record  rel_soft = relative_l2(theta_hat, theta_ref)
#           max_soft = max_abs_error(theta_hat, theta_ref)
#
#   Also record the wall error, which is the thing soft enforcement is allowed
#   to get wrong:
#           wall = boundary_points(200, pb.DOMAIN, seed=7)
#           wall_soft = max |model_soft(wall)|      in kelvin

raise NotImplementedError("Score the soft-enforcement model")

In [ ]:
print(f"  relative L2      : {rel_soft:.4e}")
print(f"  worst point      : {max_soft:.4f} K")
print(f"  worst wall error : {wall_soft:.4f} K     <- this should be zero")
print(f"  peak temperature : exact {pb.DELTA_T:.2f} K,"
      f" model {theta_hat.max():.2f} K")

fig, axes = plt.subplots(1, 2, figsize=(9.4, 6.2))
pb.plot_field(theta_hat, ax=axes[0], title="θ̂ — soft enforcement")
pb.plot_error(theta_hat, ax=axes[1])
plt.tight_layout(); plt.show()

**What you should see.** A field that looks right, a relative error of order
1e-3, and a **wall error that is small but not zero** — a few hundredths of a
kelvin, perhaps more.

That last number is the entire subject of this notebook. You told the network
the walls sit at the iron temperature. It has decided that being slightly wrong
there is worth it, because that buys a smaller PDE residual elsewhere, and the
loss you wrote said the trade was allowed.

Look at the error map. The error is rarely uniform; it usually collects near
the boundary, and the boundary is where you had information and gave it away.

---

## 5 · The weight you now have to choose

`w` was set to 1 without argument. It controls how much the walls matter
relative to the physics, and there is no principled value — which is the honest
weakness of soft enforcement.

### Your turn

In [ ]:
# TODO: sweep the penalty weight.
#
#   WEIGHTS = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
#
#   For each w: set_seed(88), a fresh MLP(2, n_hidden=32, n_layers=4), the same
#   points, train_two_stage with adam_steps=1500, lbfgs_steps=80,
#   report_every=0 to keep the output quiet.
#
#   Record for each:  relative L2 on the grid, and the worst wall error in K.
#
#   Put them in sweep = {"rel": [...], "wall": [...]}, aligned with WEIGHTS.
#
# This is six training runs and takes a couple of minutes.

raise NotImplementedError("Sweep the penalty weight")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(WEIGHTS, sweep["rel"], "o-", lw=1.9, ms=6, color="#1f77b4",
        label="relative L2 (whole field)")
ax2 = ax.twinx()
ax2.plot(WEIGHTS, sweep["wall"], "s--", lw=1.9, ms=6, color="#d94f2b",
         label="worst wall error [K]")
ax.set_xscale("log"); ax.set_yscale("log"); ax2.set_yscale("log")
ax.set_xlabel("penalty weight  w")
ax.set_ylabel("relative L2", color="#1f77b4")
ax2.set_ylabel("worst wall error [K]", color="#d94f2b")
ax.set_title("What the weight buys, and what it costs")
ax.grid(alpha=0.25, which="both")
fig.legend(frameon=False, fontsize=9, loc="upper center", ncol=2)
plt.show()

print(error_table(
    [[f"{w:g}", f"{r:.3e}", f"{b:.4f}"]
     for w, r, b in zip(WEIGHTS, sweep["rel"], sweep["wall"])],
    ["w", "relative L2", "worst wall error [K]"]))

**What you should see.** The wall error falling steadily as `w` rises — you
asked for the walls more loudly and got them. The field error, though, is
usually **U-shaped**: too small a weight and the solution drifts because the
boundary is barely constrained; too large and the boundary term dominates the
loss, the PDE gets less attention, and the interior suffers.

So there is a best `w`, it is problem-dependent, and **you found it by knowing
the exact solution.** On a problem worth solving you would not have that. This
is the real cost of soft enforcement: a hyperparameter you cannot tune honestly.

Notebook 02 removes it.

---

## 6 · Save

In [ ]:
os.makedirs("Ex07.1_outputs", exist_ok=True)
path = os.path.join("Ex07.1_outputs", "nb01_soft.npz")
np.savez(path,
         rel=rel_soft, max_err=max_soft, wall=wall_soft,
         theta_hat=theta_hat,
         weights=np.asarray(WEIGHTS),
         sweep_rel=np.asarray(sweep["rel"], dtype=float),
         sweep_wall=np.asarray(sweep["wall"], dtype=float),
         adam=history_soft["adam"], lbfgs=history_soft["lbfgs"])
torch.save(model_soft.state_dict(), os.path.join("Ex07.1_outputs", "nb01_soft.pt"))
print("wrote", path)

## 7 · Before you move on

1. The wall error came out small but non-zero. Explain, in terms of the loss
   you wrote, why the network chose that.
2. Why was the physics residual divided by the peak source before squaring?
   What would `w = 1` have meant without it?
3. You found a best `w` by comparing against the exact solution. Describe what
   you would do on a problem where no exact solution exists — and be honest
   about how much worse that is.
4. The collocation-to-parameter ratio was printed at the start of training.
   What failure would a ratio below 1 allow, and why would the loss not reveal
   it?

Next: **notebook 02**, where the boundary condition is built into the model and
cannot be violated at all.